# Imaging Confound Check — Grad-CAM / Segmentation-Overlap

In [1]:
import sys
sys.path.insert(0, "/workspace")
from utils.config import (
    CACHE_IMAGES_PATH, CACHE_MASKS_PATH, CACHE_MANIFEST_PATH, CACHE_META_PATH,
    CHECKPOINTS_IMAGING_CANDIDATES_DIR, IMAGING_OOF_PREDICTIONS_PATH,
    RESULTS_IMAGING_DIR, EVAL_IMAGING_DIR, ensure_dirs,
)
from imaging.models import ResNet50UNet, DetectionOnlyWrapper
from imaging.slice_cache_dataset import SliceCacheDataset

import json

import numpy as np
import pandas as pd
import torch
import matplotlib
matplotlib.use("Agg")  # headless -- panels are saved to disk, not displayed inline
import matplotlib.pyplot as plt
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.model_targets import RawScoresOutputTarget

ensure_dirs()

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"torch {torch.__version__}, cuda available: {torch.cuda.is_available()}, device: {DEVICE}")

with open(CACHE_META_PATH) as f:
    cache_meta = json.load(f)
BOX_SIZE = cache_meta["box_size"]
print(f"BOX_SIZE: {BOX_SIZE}")

torch 2.8.0+cu128, cuda available: True, device: cuda
BOX_SIZE: 320


## Configuration

In [2]:
CANDIDATE = "resnet50_unet"  # the winning candidate -- the one that would actually be promoted/used
FOLDS = list(range(5))

N_MSD_PER_FOLD = 40   # correctly-classified cancer slices sampled per fold (quantitative overlap metric)
N_NIH_PER_FOLD = 20   # correctly-classified healthy slices sampled per fold (border-artifact check)
N_VISUAL_PER_FOLD = 3 # subset of each, saved as visual overlay panels

TOP_ATTENTION_FRACTION = 0.20  # "attended region" = top 20% of Grad-CAM heatmap by value
BORDER_MARGIN_FRAC = 0.10      # outer 10% of image width/height counts as "border"

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

print(f"Candidate: {CANDIDATE}, folds: {FOLDS}")
print(f"MSD samples/fold: {N_MSD_PER_FOLD}, NIH samples/fold: {N_NIH_PER_FOLD}, visual/fold: {N_VISUAL_PER_FOLD}")
print(f"Top-attention fraction: {TOP_ATTENTION_FRACTION}, border margin: {BORDER_MARGIN_FRAC}")

Candidate: resnet50_unet, folds: [0, 1, 2, 3, 4]
MSD samples/fold: 40, NIH samples/fold: 20, visual/fold: 3
Top-attention fraction: 0.2, border margin: 0.1


## Load OOF Predictions, Recover Cache Row Indices

In [3]:
oof_df = pd.read_csv(IMAGING_OOF_PREDICTIONS_PATH)
oof_df = oof_df[oof_df["candidate"] == CANDIDATE].copy()
print(f"{CANDIDATE} OOF rows: {len(oof_df):,}")

manifest_df = pd.read_csv(CACHE_MANIFEST_PATH)
oof_df = oof_df.merge(
    manifest_df[["patient_id", "slice_index", "dataset", "img_row", "mask_row", "class"]],
    on=["patient_id", "slice_index", "dataset"], how="left", validate="one_to_one",
)
assert oof_df["img_row"].notna().all(), "every OOF row must join back to a manifest row"
print("Join OK -- img_row/mask_row recovered for every OOF row")

resnet50_unet OOF rows: 90,693


Join OK -- img_row/mask_row recovered for every OOF row


## Model + Grad-CAM Loading

In [4]:
def load_fold_model(fold: int):
    """Loads the checkpoint HELD OUT on this fold. Returns (wrapped_model, base_model) --
    base_model is needed separately to reference layer4[-1] as the Grad-CAM target layer."""
    ckpt_path = CHECKPOINTS_IMAGING_CANDIDATES_DIR / CANDIDATE / f"fold_{fold}.pt"
    model = ResNet50UNet(pretrained=False)
    model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE, weights_only=True))
    model = model.to(DEVICE).eval()
    return DetectionOnlyWrapper(model), model


print("load_fold_model defined")

load_fold_model defined


## Attention Metrics

In [5]:
def compute_attention_metrics(heatmap: np.ndarray, foreground_mask: np.ndarray = None) -> dict:
    """heatmap: (H,W) Grad-CAM output in [0,1]. foreground_mask: (H,W) bool, True where real
    pancreas/tumour anatomy is -- None for NIH rows (no ground-truth mask exists)."""
    h, w = heatmap.shape
    threshold = np.percentile(heatmap, 100 * (1 - TOP_ATTENTION_FRACTION))
    attended = heatmap >= threshold

    margin_h, margin_w = max(int(h * BORDER_MARGIN_FRAC), 1), max(int(w * BORDER_MARGIN_FRAC), 1)
    border_mask = np.zeros((h, w), dtype=bool)
    border_mask[:margin_h, :] = True
    border_mask[-margin_h:, :] = True
    border_mask[:, :margin_w] = True
    border_mask[:, -margin_w:] = True
    border_fraction = float((attended & border_mask).sum() / max(attended.sum(), 1))

    result = {"border_fraction": border_fraction, "attended_pixel_count": int(attended.sum())}

    if foreground_mask is not None and foreground_mask.sum() > 0:
        overlap = attended & foreground_mask
        result["overlap_precision"] = float(overlap.sum() / max(attended.sum(), 1))
        result["overlap_recall"] = float(overlap.sum() / foreground_mask.sum())
        chance_precision = float(foreground_mask.sum() / (h * w))
        result["chance_precision"] = chance_precision
        result["enrichment"] = result["overlap_precision"] / chance_precision if chance_precision > 0 else float("nan")

    return result


print("compute_attention_metrics defined")

compute_attention_metrics defined


## Visual Panels

In [6]:
def save_visual_panel(fold, msd_dataset, msd_heatmaps, nih_dataset, nih_heatmaps, n_visual=N_VISUAL_PER_FOLD):
    """One PNG per fold: n_visual MSD examples + n_visual NIH examples, image alongside its
    Grad-CAM overlay (mask contour in lime for MSD, where a real mask exists)."""
    n_msd = min(n_visual, len(msd_dataset))
    n_nih = min(n_visual, len(nih_dataset))
    n_rows = n_msd + n_nih
    fig, axes = plt.subplots(n_rows, 2, figsize=(6, 3 * n_rows))
    if n_rows == 1:
        axes = axes[None, :]

    for i in range(n_msd):
        img = msd_dataset[i]["image"][0].numpy()
        mask = msd_dataset[i]["mask"].numpy()
        axes[i, 0].imshow(img, cmap="gray")
        axes[i, 0].set_title(f"MSD cancer (fold {fold})")
        axes[i, 0].axis("off")
        axes[i, 1].imshow(img, cmap="gray")
        axes[i, 1].imshow(msd_heatmaps[i], cmap="jet", alpha=0.45)
        axes[i, 1].contour(mask >= 1, levels=[0.5], colors="lime", linewidths=1)
        axes[i, 1].set_title("Grad-CAM + real mask (lime)")
        axes[i, 1].axis("off")

    for j in range(n_nih):
        r = n_msd + j
        img = nih_dataset[j]["image"][0].numpy()
        axes[r, 0].imshow(img, cmap="gray")
        axes[r, 0].set_title(f"NIH healthy (fold {fold})")
        axes[r, 0].axis("off")
        axes[r, 1].imshow(img, cmap="gray")
        axes[r, 1].imshow(nih_heatmaps[j], cmap="jet", alpha=0.45)
        axes[r, 1].set_title("Grad-CAM (no mask -- NIH unlabeled)")
        axes[r, 1].axis("off")

    plt.tight_layout()
    out_path = EVAL_IMAGING_DIR / f"gradcam_confound_check_fold{fold}.png"
    fig.savefig(out_path, dpi=110)
    plt.close(fig)
    print(f"Saved {out_path}")


print("save_visual_panel defined")

save_visual_panel defined


## Per-Fold Run

In [7]:
def run_confound_check_for_fold(fold: int) -> dict:
    """Grad-CAM + metrics for one fold's held-out, correctly-classified MSD and NIH slices."""
    wrapped_model, base_model = load_fold_model(fold)

    fold_oof = oof_df[oof_df["fold"] == fold]
    msd_pool = fold_oof[(fold_oof["dataset"] == "MSD") & (fold_oof["y_true"] == 1) & (fold_oof["y_pred"] == 1)]
    nih_pool = fold_oof[(fold_oof["dataset"] == "NIH") & (fold_oof["y_true"] == 0) & (fold_oof["y_pred"] == 0)]

    msd_sample = msd_pool.sample(n=min(N_MSD_PER_FOLD, len(msd_pool)), random_state=RANDOM_SEED).reset_index(drop=True)
    nih_sample = nih_pool.sample(n=min(N_NIH_PER_FOLD, len(nih_pool)), random_state=RANDOM_SEED).reset_index(drop=True)

    msd_dataset = SliceCacheDataset(msd_sample, CACHE_IMAGES_PATH, CACHE_MASKS_PATH, True, BOX_SIZE)
    nih_dataset = SliceCacheDataset(nih_sample, CACHE_IMAGES_PATH, CACHE_MASKS_PATH, True, BOX_SIZE)

    msd_images = torch.stack([msd_dataset[i]["image"] for i in range(len(msd_dataset))]).to(DEVICE)
    nih_images = torch.stack([nih_dataset[i]["image"] for i in range(len(nih_dataset))]).to(DEVICE)

    with GradCAM(model=wrapped_model, target_layers=[base_model.layer4[-1]]) as cam:
        msd_heatmaps = cam(input_tensor=msd_images, targets=[RawScoresOutputTarget()] * len(msd_images))
        nih_heatmaps = cam(input_tensor=nih_images, targets=[RawScoresOutputTarget()] * len(nih_images))

    msd_rows, nih_rows = [], []
    for i in range(len(msd_dataset)):
        mask = msd_dataset[i]["mask"].numpy()
        metrics = compute_attention_metrics(msd_heatmaps[i], mask >= 1)
        metrics.update({"fold": fold, "dataset": "MSD",
                         "patient_id": msd_sample.iloc[i]["patient_id"],
                         "slice_index": int(msd_sample.iloc[i]["slice_index"])})
        msd_rows.append(metrics)

    for j in range(len(nih_dataset)):
        metrics = compute_attention_metrics(nih_heatmaps[j], None)
        metrics.update({"fold": fold, "dataset": "NIH",
                         "patient_id": nih_sample.iloc[j]["patient_id"],
                         "slice_index": int(nih_sample.iloc[j]["slice_index"])})
        nih_rows.append(metrics)

    save_visual_panel(fold, msd_dataset, msd_heatmaps, nih_dataset, nih_heatmaps)
    print(f"[fold {fold}] MSD n={len(msd_rows)}, NIH n={len(nih_rows)} -- Grad-CAM + metrics computed")

    del wrapped_model, base_model
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

    return {"msd_rows": msd_rows, "nih_rows": nih_rows}


print("run_confound_check_for_fold defined")

run_confound_check_for_fold defined


In [8]:
all_msd_rows, all_nih_rows = [], []
for fold in FOLDS:
    result = run_confound_check_for_fold(fold)
    all_msd_rows.extend(result["msd_rows"])
    all_nih_rows.extend(result["nih_rows"])

msd_metrics_df = pd.DataFrame(all_msd_rows)
nih_metrics_df = pd.DataFrame(all_nih_rows)
print(f"\nTotal: {len(msd_metrics_df)} MSD slices, {len(nih_metrics_df)} NIH slices probed across {len(FOLDS)} folds")

Saved /workspace/outputs/eval/imaging/gradcam_confound_check_fold0.png
[fold 0] MSD n=40, NIH n=20 -- Grad-CAM + metrics computed


Saved /workspace/outputs/eval/imaging/gradcam_confound_check_fold1.png
[fold 1] MSD n=40, NIH n=20 -- Grad-CAM + metrics computed


Saved /workspace/outputs/eval/imaging/gradcam_confound_check_fold2.png
[fold 2] MSD n=40, NIH n=20 -- Grad-CAM + metrics computed


Saved /workspace/outputs/eval/imaging/gradcam_confound_check_fold3.png
[fold 3] MSD n=40, NIH n=20 -- Grad-CAM + metrics computed


Saved /workspace/outputs/eval/imaging/gradcam_confound_check_fold4.png
[fold 4] MSD n=40, NIH n=20 -- Grad-CAM + metrics computed

Total: 200 MSD slices, 100 NIH slices probed across 5 folds


## Aggregate, Save, and State a Verdict

In [9]:
combined_df = pd.concat(
    [msd_metrics_df.assign(dataset="MSD"), nih_metrics_df.assign(dataset="NIH")],
    ignore_index=True, sort=False,
)
out_csv = RESULTS_IMAGING_DIR / "confound_check_overlap_metrics.csv"
combined_df.to_csv(out_csv, index=False)
print(f"Wrote {out_csv} ({len(combined_df)} rows)")

mean_overlap_precision = msd_metrics_df["overlap_precision"].mean()
mean_overlap_recall = msd_metrics_df["overlap_recall"].mean()
mean_chance_precision = msd_metrics_df["chance_precision"].mean()
mean_enrichment = msd_metrics_df["enrichment"].mean()
mean_msd_border = msd_metrics_df["border_fraction"].mean()
mean_nih_border = nih_metrics_df["border_fraction"].mean()

print("=== Confound Check Summary ===")
print(f"MSD (cancer, n={len(msd_metrics_df)}):")
print(f"  mean attention-anatomy overlap precision = {mean_overlap_precision:.3f}")
print(f"  vs. chance-level precision (mask area fraction) = {mean_chance_precision:.3f}")
print(f"  enrichment factor = {mean_enrichment:.2f}x (>1 means attention concentrates on real anatomy more than random)")
print(f"  mean overlap recall (fraction of real anatomy covered by top attention) = {mean_overlap_recall:.3f}")
print(f"  mean border_fraction (top attention in outer {BORDER_MARGIN_FRAC:.0%} margin) = {mean_msd_border:.3f}")
print(f"NIH (healthy, n={len(nih_metrics_df)}):")
print(f"  mean border_fraction = {mean_nih_border:.3f} (no ground-truth mask -- border check only)")

print("\n=== Verdict ===")
VERDICT_ENRICHMENT_THRESHOLD = 1.5
VERDICT_BORDER_THRESHOLD = 0.30
if mean_enrichment > VERDICT_ENRICHMENT_THRESHOLD and mean_msd_border < VERDICT_BORDER_THRESHOLD and mean_nih_border < VERDICT_BORDER_THRESHOLD:
    print("No evidence of the scanner/acquisition confound: Grad-CAM attention on correctly-classified "
          f"cancer slices concentrates on real pancreas/tumour anatomy at {mean_enrichment:.2f}x the rate "
          "chance would predict, and attention is not disproportionately piling up at image borders on "
          "either dataset. This does not prove the model never uses scanner cues at all -- it means this "
          "specific check did not catch the specific failure mode it was designed to catch.")
else:
    print("POTENTIAL CONFOUND SIGNAL -- reporting honestly per the original instruction. "
          f"enrichment={mean_enrichment:.2f}x, MSD border_fraction={mean_msd_border:.3f}, "
          f"NIH border_fraction={mean_nih_border:.3f}. Inspect the saved panels in "
          f"{EVAL_IMAGING_DIR} before trusting the detection head's decisions.")

Wrote /workspace/results/imaging/confound_check_overlap_metrics.csv (300 rows)
=== Confound Check Summary ===
MSD (cancer, n=200):
  mean attention-anatomy overlap precision = 0.000
  vs. chance-level precision (mask area fraction) = 0.009
  enrichment factor = 0.00x (>1 means attention concentrates on real anatomy more than random)
  mean overlap recall (fraction of real anatomy covered by top attention) = 0.000
  mean border_fraction (top attention in outer 10% margin) = 0.209
NIH (healthy, n=100):
  mean border_fraction = 0.294 (no ground-truth mask -- border check only)

=== Verdict ===
POTENTIAL CONFOUND SIGNAL -- reporting honestly per the original instruction. enrichment=0.00x, MSD border_fraction=0.209, NIH border_fraction=0.294. Inspect the saved panels in /workspace/outputs/eval/imaging before trusting the detection head's decisions.


## Cross-Check with a Second Attribution Method (Integrated Gradients)

In [10]:
from captum.attr import IntegratedGradients

IG_N_STEPS = 20  # integration steps -- 20 is captum's typical default range, sufficient for a summary statistic


def compute_ig_heatmap(wrapped_model, image_tensor: torch.Tensor) -> np.ndarray:
    """Integrated Gradients attribution for one image (1,3,H,W), collapsed across channels
    (sum of abs -- direction doesn't matter here, only where attribution concentrates) and
    min-max normalized to [0,1] so it's directly comparable to a Grad-CAM heatmap via the
    same compute_attention_metrics() function."""
    ig = IntegratedGradients(wrapped_model)
    baseline = torch.zeros_like(image_tensor)
    attributions = ig.attribute(image_tensor, baselines=baseline, target=None, n_steps=IG_N_STEPS)
    attr = attributions[0].abs().sum(dim=0).detach().cpu().numpy()
    attr = (attr - attr.min()) / (attr.max() - attr.min() + 1e-8)
    return attr


# Re-select exactly the MSD rows that had a real mask in the Grad-CAM pass above --
# same slices, same fold-appropriate checkpoints, direct paired comparison.
masked_msd_rows = msd_metrics_df[msd_metrics_df["overlap_precision"].notna()].copy()
print(f"Cross-checking {len(masked_msd_rows)} MSD slices with real masks (same slices as the Grad-CAM pass)")

ig_rows = []
for fold in FOLDS:
    fold_rows = masked_msd_rows[masked_msd_rows["fold"] == fold]
    if len(fold_rows) == 0:
        continue
    wrapped_model, base_model = load_fold_model(fold)

    lookup = fold_rows.merge(oof_df, on=["fold", "patient_id", "slice_index"], how="left", suffixes=("", "_oof"))
    ds = SliceCacheDataset(lookup.reset_index(drop=True), CACHE_IMAGES_PATH, CACHE_MASKS_PATH, True, BOX_SIZE)

    for i in range(len(ds)):
        item = ds[i]
        image_tensor = item["image"].unsqueeze(0).to(DEVICE)
        mask = item["mask"].numpy()
        ig_heatmap = compute_ig_heatmap(wrapped_model, image_tensor)
        metrics = compute_attention_metrics(ig_heatmap, mask >= 1)
        metrics.update({"fold": fold, "patient_id": lookup.iloc[i]["patient_id"],
                         "slice_index": int(lookup.iloc[i]["slice_index"]), "method": "integrated_gradients"})
        ig_rows.append(metrics)

    del wrapped_model, base_model
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    print(f"[fold {fold}] Integrated Gradients: {len(fold_rows)} slices done")

ig_metrics_df = pd.DataFrame(ig_rows)
print(f"\nTotal IG cross-check: {len(ig_metrics_df)} slices")

Cross-checking 65 MSD slices with real masks (same slices as the Grad-CAM pass)


[fold 0] Integrated Gradients: 10 slices done


[fold 1] Integrated Gradients: 16 slices done


[fold 2] Integrated Gradients: 14 slices done


[fold 3] Integrated Gradients: 13 slices done


[fold 4] Integrated Gradients: 12 slices done

Total IG cross-check: 65 slices


## Paired Comparison and Final Verdict

In [11]:
ig_out_csv = RESULTS_IMAGING_DIR / "confound_check_integrated_gradients_metrics.csv"
ig_metrics_df.to_csv(ig_out_csv, index=False)
print(f"Wrote {ig_out_csv} ({len(ig_metrics_df)} rows)")

gradcam_paired = masked_msd_rows.set_index(["fold", "patient_id", "slice_index"])
ig_paired = ig_metrics_df.set_index(["fold", "patient_id", "slice_index"])

print("=== Paired comparison (same 59 slices, same checkpoints) ===")
print(f"{'metric':<20}{'Grad-CAM':>12}{'Integrated Grad.':>18}")
for metric in ["overlap_precision", "overlap_recall", "enrichment", "border_fraction"]:
    gc_mean = gradcam_paired[metric].mean()
    ig_mean = ig_paired[metric].mean()
    print(f"{metric:<20}{gc_mean:>12.4f}{ig_mean:>18.4f}")

ig_enrichment = ig_paired["enrichment"].mean()
ig_border = ig_paired["border_fraction"].mean()
gc_enrichment = gradcam_paired["enrichment"].mean()

print("\n=== Final Verdict (both methods) ===")
VERDICT_ENRICHMENT_THRESHOLD = 1.5
both_low = gc_enrichment < 1.0 and ig_enrichment < 1.0
if both_low:
    print(f"CONFOUND SIGNAL CONFIRMED BY BOTH METHODS. Grad-CAM enrichment={gc_enrichment:.3f}x, "
          f"Integrated Gradients enrichment={ig_enrichment:.3f}x -- both well below 1x (chance). "
          "Two attribution methods with different, unrelated failure modes producing the same "
          "'attention does not concentrate on real pancreas/tumour anatomy' result is strong "
          "evidence this reflects what the detection head actually learned, not an artifact of "
          "either method. This does NOT prove specifically that dataset/scanner identity is the "
          "shortcut being used (that would need e.g. deliberately swapping scanner metadata while "
          "holding anatomy fixed, which this dataset's structure does not allow to test cleanly) -- "
          "but it does mean the detection head's near-perfect ROC-AUC should NOT be read as evidence "
          "the model is reasoning about pathology. Recommend disclosing this explicitly in the "
          "write-up as a limitation of the imaging branch, not silently promoting resnet50_unet as "
          "'validated' on the strength of its detection metrics alone."
    )
elif ig_enrichment > VERDICT_ENRICHMENT_THRESHOLD:
    print(f"MIXED RESULT. Grad-CAM enrichment={gc_enrichment:.3f}x (near-zero) but Integrated Gradients "
          f"enrichment={ig_enrichment:.3f}x (concentrates on real anatomy). This is consistent with "
          "Grad-CAM's known conv-padding boundary artifact rather than a genuine model confound -- "
          "the model does appear to attend to real anatomy under a method not subject to that specific "
          "artifact. Recommend trusting Integrated Gradients over Grad-CAM for this architecture, and "
          "noting Grad-CAM's unreliability here rather than the model's, in the write-up."
    )
else:
    print(f"Grad-CAM enrichment={gc_enrichment:.3f}x, Integrated Gradients enrichment={ig_enrichment:.3f}x -- "
          "neither cleanly confirms nor rules out the confound. Report both numbers as-is; treat the "
          "detection head's real-world reliability as unresolved pending further investigation "
          "(e.g. occlusion-based sensitivity analysis, or scanner-metadata ablation if feasible).")

Wrote /workspace/results/imaging/confound_check_integrated_gradients_metrics.csv (65 rows)
=== Paired comparison (same 59 slices, same checkpoints) ===
metric                  Grad-CAM  Integrated Grad.
overlap_precision         0.0000            0.0146
overlap_recall            0.0000            0.3103
enrichment                0.0000            1.5514
border_fraction           0.1995            0.1040

=== Final Verdict (both methods) ===
MIXED RESULT. Grad-CAM enrichment=0.000x (near-zero) but Integrated Gradients enrichment=1.551x (concentrates on real anatomy). This is consistent with Grad-CAM's known conv-padding boundary artifact rather than a genuine model confound -- the model does appear to attend to real anatomy under a method not subject to that specific artifact. Recommend trusting Integrated Gradients over Grad-CAM for this architecture, and noting Grad-CAM's unreliability here rather than the model's, in the write-up.


## Cross-Check with Four More Attribution Methods

In [12]:
from pytorch_grad_cam import GradCAMPlusPlus, HiResCAM, XGradCAM, EigenCAM


def run_cam_family_method(cam_class, method_name: str) -> pd.DataFrame:
    """Runs a pytorch-grad-cam family method (targeting layer4[-1]) on the same 59 MSD
    slices with real masks, same fold-appropriate checkpoints, as every other method in this
    notebook -- for a fully paired comparison."""
    rows = []
    for fold in FOLDS:
        fold_rows = masked_msd_rows[masked_msd_rows["fold"] == fold]
        if len(fold_rows) == 0:
            continue
        wrapped_model, base_model = load_fold_model(fold)
        lookup = fold_rows.merge(oof_df, on=["fold", "patient_id", "slice_index"], how="left", suffixes=("", "_oof"))
        ds = SliceCacheDataset(lookup.reset_index(drop=True), CACHE_IMAGES_PATH, CACHE_MASKS_PATH, True, BOX_SIZE)
        images = torch.stack([ds[i]["image"] for i in range(len(ds))]).to(DEVICE)

        with cam_class(model=wrapped_model, target_layers=[base_model.layer4[-1]]) as cam:
            heatmaps = cam(input_tensor=images, targets=[RawScoresOutputTarget()] * len(images))

        for i in range(len(ds)):
            mask = ds[i]["mask"].numpy()
            metrics = compute_attention_metrics(heatmaps[i], mask >= 1)
            metrics.update({"fold": fold, "patient_id": lookup.iloc[i]["patient_id"],
                             "slice_index": int(lookup.iloc[i]["slice_index"]), "method": method_name})
            rows.append(metrics)

        del wrapped_model, base_model
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()

    df = pd.DataFrame(rows)
    print(f"{method_name}: {len(df)} slices, mean enrichment={df['enrichment'].mean():.3f}, "
          f"mean overlap_recall={df['overlap_recall'].mean():.3f}, mean border_fraction={df['border_fraction'].mean():.3f}")
    return df


gradcam_pp_df = run_cam_family_method(GradCAMPlusPlus, "gradcam_plusplus")
hirescam_df = run_cam_family_method(HiResCAM, "hirescam")
xgradcam_df = run_cam_family_method(XGradCAM, "xgradcam")
eigencam_df = run_cam_family_method(EigenCAM, "eigencam")

gradcam_plusplus: 65 slices, mean enrichment=0.000, mean overlap_recall=0.000, mean border_fraction=0.206


hirescam: 65 slices, mean enrichment=0.000, mean overlap_recall=0.000, mean border_fraction=0.199


xgradcam: 65 slices, mean enrichment=0.000, mean overlap_recall=0.000, mean border_fraction=0.199


eigencam: 65 slices, mean enrichment=0.087, mean overlap_recall=0.042, mean border_fraction=0.296


In [13]:
gradcam_df = masked_msd_rows.copy()
gradcam_df["method"] = "gradcam"
ig_df = ig_metrics_df.copy()
ig_df["method"] = "integrated_gradients"

all_methods_df = pd.concat(
    [gradcam_df, gradcam_pp_df, hirescam_df, xgradcam_df, eigencam_df, ig_df],
    ignore_index=True, sort=False,
)
all_methods_out_csv = RESULTS_IMAGING_DIR / "confound_check_all_methods_metrics.csv"
all_methods_df.to_csv(all_methods_out_csv, index=False)
print(f"Wrote {all_methods_out_csv} ({len(all_methods_df)} rows, {all_methods_df['method'].nunique()} methods)")

summary = all_methods_df.groupby("method")[["enrichment", "overlap_recall", "border_fraction"]].mean()
summary = summary.sort_values("enrichment", ascending=False)
print("\n=== 6-Method Summary (same 59 slices, same checkpoints) ===")
print(summary.round(4).to_string())

n_methods = len(summary)
n_below_chance = (summary["enrichment"] < 0.5).sum()
n_at_or_above_chance = (summary["enrichment"] >= 0.5).sum()
print(f"\n{n_below_chance}/{n_methods} methods: enrichment < 0.5x (well below chance -- attention avoids real anatomy)")
print(f"{n_at_or_above_chance}/{n_methods} methods: enrichment >= 0.5x (at or above chance)")

Wrote /workspace/results/imaging/confound_check_all_methods_metrics.csv (390 rows, 6 methods)

=== 6-Method Summary (same 59 slices, same checkpoints) ===
                      enrichment  overlap_recall  border_fraction
method                                                           
integrated_gradients      1.5514          0.3103           0.1040
eigencam                  0.0871          0.0420           0.2959
gradcam                   0.0000          0.0000           0.1995
gradcam_plusplus          0.0000          0.0000           0.2064
hirescam                  0.0000          0.0000           0.1995
xgradcam                  0.0000          0.0000           0.1995

5/6 methods: enrichment < 0.5x (well below chance -- attention avoids real anatomy)
1/6 methods: enrichment >= 0.5x (at or above chance)


In [14]:
def save_multi_method_panel(example_rows: pd.DataFrame, n_examples: int = 3):
    """Side-by-side visual comparison of all 6 methods on the same few example slices --
    the numbers above are the rigorous version of this, but seeing all six heatmaps on one
    image at a glance is often what actually catches a subtle pattern the numbers smooth over."""
    method_classes = {"gradcam": GradCAM, "gradcam_plusplus": GradCAMPlusPlus,
                       "hirescam": HiResCAM, "xgradcam": XGradCAM, "eigencam": EigenCAM}
    examples = example_rows.head(n_examples).reset_index(drop=True)

    fig, axes = plt.subplots(n_examples, 7, figsize=(21, 3 * n_examples))
    if n_examples == 1:
        axes = axes[None, :]

    for row_i in range(len(examples)):
        ex = examples.iloc[row_i]
        wrapped_model, base_model = load_fold_model(int(ex["fold"]))
        lookup = pd.DataFrame([ex]).merge(oof_df, on=["fold", "patient_id", "slice_index"], how="left", suffixes=("", "_oof"))
        ds = SliceCacheDataset(lookup, CACHE_IMAGES_PATH, CACHE_MASKS_PATH, True, BOX_SIZE)
        item = ds[0]
        img = item["image"][0].numpy()
        mask = item["mask"].numpy()
        image_tensor = item["image"].unsqueeze(0).to(DEVICE)

        axes[row_i, 0].imshow(img, cmap="gray")
        axes[row_i, 0].contour(mask >= 1, levels=[0.5], colors="lime", linewidths=1.5)
        axes[row_i, 0].set_title(f"original + real mask\nfold{int(ex['fold'])} {ex['patient_id']}")
        axes[row_i, 0].axis("off")

        for col_i, (name, cam_cls) in enumerate(method_classes.items(), start=1):
            with cam_cls(model=wrapped_model, target_layers=[base_model.layer4[-1]]) as cam:
                hm = cam(input_tensor=image_tensor, targets=[RawScoresOutputTarget()])[0]
            axes[row_i, col_i].imshow(img, cmap="gray")
            axes[row_i, col_i].imshow(hm, cmap="jet", alpha=0.45)
            axes[row_i, col_i].contour(mask >= 1, levels=[0.5], colors="lime", linewidths=1)
            axes[row_i, col_i].set_title(name)
            axes[row_i, col_i].axis("off")

        ig_hm = compute_ig_heatmap(wrapped_model, image_tensor)
        axes[row_i, 6].imshow(img, cmap="gray")
        axes[row_i, 6].imshow(ig_hm, cmap="jet", alpha=0.45)
        axes[row_i, 6].contour(mask >= 1, levels=[0.5], colors="lime", linewidths=1)
        axes[row_i, 6].set_title("integrated_gradients")
        axes[row_i, 6].axis("off")

        del wrapped_model, base_model
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()

    plt.tight_layout()
    out_path = EVAL_IMAGING_DIR / "confound_check_all_methods_comparison.png"
    fig.savefig(out_path, dpi=110)
    plt.close(fig)
    print(f"Saved {out_path}")


save_multi_method_panel(masked_msd_rows.sample(n=3, random_state=RANDOM_SEED))

Saved /workspace/outputs/eval/imaging/confound_check_all_methods_comparison.png


In [15]:
print("=== FINAL VERDICT: 6 attribution methods, same 59 slices, same checkpoints ===\n")
print(summary.round(4).to_string())

gradient_methods = ["gradcam", "gradcam_plusplus", "hirescam", "xgradcam"]
gradient_mean_enrichment = summary.loc[summary.index.isin(gradient_methods), "enrichment"].mean()
eigencam_enrichment = summary.loc["eigencam", "enrichment"] if "eigencam" in summary.index else float("nan")
ig_enrichment_final = summary.loc["integrated_gradients", "enrichment"] if "integrated_gradients" in summary.index else float("nan")

print(f"\nGradient-based methods through layer4 (gradcam, gradcam++, hirescam, xgradcam) mean enrichment: {gradient_mean_enrichment:.3f}x")
print(f"EigenCAM (gradient-free, same layer) enrichment: {eigencam_enrichment:.3f}x")
print(f"Integrated Gradients (input-space, different layer entirely) enrichment: {ig_enrichment_final:.3f}x")

print("\n=== Interpretation ===")
if gradient_mean_enrichment < 0.5 and eigencam_enrichment < 0.5:
    print("ALL layer4-based methods (gradient and gradient-free alike) agree: near/below-chance "
          "overlap with real anatomy. Since EigenCAM needs no gradients at all and still shows "
          "the same pattern, this rules out 'it's a gradient-computation artifact' as the "
          "explanation -- the pattern is a property of what layer4's activations respond to for "
          "this input, not an artifact of backpropagating through padded convolutions. "
          "Integrated Gradients (input-space, not layer4) is the outlier here, not the majority.")
elif gradient_mean_enrichment < 0.5 and eigencam_enrichment >= 0.5:
    print("Gradient-based layer4 methods show near/below-chance overlap, but EigenCAM "
          "(gradient-free, same layer) does not. This DOES support the boundary-artifact "
          "explanation specifically -- the pattern is tied to how gradients propagate through "
          "layer4, not to what layer4's activations actually encode. Recommend trusting "
          "EigenCAM/Integrated Gradients over the gradient-based CAM family for this "
          "architecture, and noting this as a known limitation of gradient-based CAMs on "
          "heavily-padded medical images specifically.")
else:
    print("No single, clean explanation emerges across all 6 methods -- report the full table "
          "above as-is rather than force a summary verdict past what the evidence supports.")

print("\nRegardless of which explanation above applies, the practical conclusion is unchanged: "
      "no method here showed confident (enrichment clearly > 1) concentration on the real "
      "tumour/pancreas region, so the 0.994 ROC-AUC should not be presented as evidence of "
      "pixel-level pathological reasoning without this caveat attached.")

=== FINAL VERDICT: 6 attribution methods, same 59 slices, same checkpoints ===

                      enrichment  overlap_recall  border_fraction
method                                                           
integrated_gradients      1.5514          0.3103           0.1040
eigencam                  0.0871          0.0420           0.2959
gradcam                   0.0000          0.0000           0.1995
gradcam_plusplus          0.0000          0.0000           0.2064
hirescam                  0.0000          0.0000           0.1995
xgradcam                  0.0000          0.0000           0.1995

Gradient-based methods through layer4 (gradcam, gradcam++, hirescam, xgradcam) mean enrichment: 0.000x
EigenCAM (gradient-free, same layer) enrichment: 0.087x
Integrated Gradients (input-space, different layer entirely) enrichment: 1.551x

=== Interpretation ===
ALL layer4-based methods (gradient and gradient-free alike) agree: near/below-chance overlap with real anatomy. Since EigenCAM n

## Occlusion-Based Sensitivity Analysis

In [16]:
def detect_padding_margins(img: np.ndarray) -> tuple:
    """Largest contiguous exact-zero margin on each side (top, bottom, left, right) -- same
    detector used in the earlier packing investigation. Real anatomy (even dark/air regions)
    has pixel-level noise/texture; only synthetic padding is bit-for-bit uniform zero."""
    h, w = img.shape

    def margin(fn, n):
        for k in range(n):
            if not np.all(fn(k) == 0):
                return k
        return n

    top = margin(lambda k: img[k, :], h // 2)
    bottom = margin(lambda k: img[h - 1 - k, :], h // 2)
    left = margin(lambda k: img[:, k], w // 2)
    right = margin(lambda k: img[:, w - 1 - k], w // 2)
    return top, bottom, left, right


def occlude_region(image_3ch: torch.Tensor, region_mask: np.ndarray, fill_value: float) -> torch.Tensor:
    """Replaces pixels where region_mask is True with fill_value across all 3 (identical)
    channels. Returns a new tensor -- never modifies the input in place."""
    occluded = image_3ch.clone()
    region_t = torch.from_numpy(region_mask)
    for c in range(occluded.shape[0]):
        occluded[c][region_t] = fill_value
    return occluded


def random_control_region(h: int, w: int, area: int, avoid_mask: np.ndarray,
                           rng: np.random.RandomState, n_tries: int = 30) -> np.ndarray:
    """A random square region with ~`area` pixels, best-effort avoiding overlap with
    avoid_mask (the tumour + padding regions) -- falls back to the last try if no clean
    placement is found in n_tries attempts."""
    side = max(int(np.sqrt(area)), 1)
    candidate = np.zeros((h, w), dtype=bool)
    for _ in range(n_tries):
        y0 = rng.randint(0, max(h - side, 1))
        x0 = rng.randint(0, max(w - side, 1))
        candidate = np.zeros((h, w), dtype=bool)
        candidate[y0:y0 + side, x0:x0 + side] = True
        if not (candidate & avoid_mask).any():
            return candidate
    return candidate


@torch.no_grad()
def get_prediction(wrapped_model, image_tensor: torch.Tensor) -> dict:
    """Both the raw logit and the sigmoid probability for a single (1,3,H,W) image tensor.
    The logit is the PRIMARY metric used below -- see the markdown note added after the first
    run of this analysis: baseline P(cancer) on these correctly-classified slices averaged
    0.999, i.e. the sigmoid is already saturated, leaving almost no room for any local
    occlusion (tumor, padding, or a random patch) to move the *probability* regardless of
    whether the occluded region actually matters. The logit is unbounded and doesn't saturate,
    so it stays sensitive to input changes even when the probability is pinned near 1.0."""
    logit = float(wrapped_model(image_tensor).item())
    return {"logit": logit, "prob": float(torch.sigmoid(torch.tensor(logit)).item())}


print("occlusion helpers defined")

occlusion helpers defined


In [17]:
MIN_PADDING_PX_TO_OCCLUDE = 100  # skip the padding-occlusion condition on slices with negligible padding

rng = np.random.RandomState(RANDOM_SEED)
occlusion_rows = []

for fold in FOLDS:
    fold_rows = masked_msd_rows[masked_msd_rows["fold"] == fold]
    if len(fold_rows) == 0:
        continue
    wrapped_model, base_model = load_fold_model(fold)
    lookup = fold_rows.merge(oof_df, on=["fold", "patient_id", "slice_index"], how="left", suffixes=("", "_oof"))
    ds = SliceCacheDataset(lookup.reset_index(drop=True), CACHE_IMAGES_PATH, CACHE_MASKS_PATH, True, BOX_SIZE)

    for i in range(len(ds)):
        item = ds[i]
        image = item["image"]  # (3,H,W), channels identical
        mask = item["mask"].numpy()
        tumor_mask = mask >= 1
        h, w = mask.shape
        fill_value = float(image[0].median())

        baseline = get_prediction(wrapped_model, image.unsqueeze(0).to(DEVICE))

        tumor_occluded = occlude_region(image, tumor_mask, fill_value)
        tumor_pred = get_prediction(wrapped_model, tumor_occluded.unsqueeze(0).to(DEVICE))

        top, bottom, left, right = detect_padding_margins(image[0].numpy())
        pad_mask = np.zeros((h, w), dtype=bool)
        pad_mask[:top, :] = True
        pad_mask[h - bottom:, :] = True
        pad_mask[:, :left] = True
        pad_mask[:, w - right:] = True
        has_meaningful_padding = bool(pad_mask.sum() >= MIN_PADDING_PX_TO_OCCLUDE)

        if has_meaningful_padding:
            padding_occluded = occlude_region(image, pad_mask, fill_value)
            padding_pred = get_prediction(wrapped_model, padding_occluded.unsqueeze(0).to(DEVICE))
        else:
            padding_pred = {"logit": float("nan"), "prob": float("nan")}

        control_mask = random_control_region(h, w, int(tumor_mask.sum()), tumor_mask | pad_mask, rng)
        control_occluded = occlude_region(image, control_mask, fill_value)
        control_pred = get_prediction(wrapped_model, control_occluded.unsqueeze(0).to(DEVICE))

        occlusion_rows.append({
            "fold": fold, "patient_id": lookup.iloc[i]["patient_id"], "slice_index": int(lookup.iloc[i]["slice_index"]),
            "baseline_logit": baseline["logit"], "baseline_p": baseline["prob"],
            "tumor_occluded_logit": tumor_pred["logit"], "tumor_occluded_p": tumor_pred["prob"],
            "padding_occluded_logit": padding_pred["logit"], "padding_occluded_p": padding_pred["prob"],
            "control_occluded_logit": control_pred["logit"], "control_occluded_p": control_pred["prob"],
            "delta_tumor_logit": baseline["logit"] - tumor_pred["logit"],
            "delta_padding_logit": (baseline["logit"] - padding_pred["logit"]) if has_meaningful_padding else float("nan"),
            "delta_control_logit": baseline["logit"] - control_pred["logit"],
            "delta_tumor_p": baseline["prob"] - tumor_pred["prob"],
            "delta_padding_p": (baseline["prob"] - padding_pred["prob"]) if has_meaningful_padding else float("nan"),
            "delta_control_p": baseline["prob"] - control_pred["prob"],
            "has_meaningful_padding": has_meaningful_padding,
            "tumor_area_px": int(tumor_mask.sum()), "padding_area_px": int(pad_mask.sum()),
        })

    del wrapped_model, base_model
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    print(f"[fold {fold}] occlusion sensitivity: {len(fold_rows)} slices done")

occlusion_df = pd.DataFrame(occlusion_rows)
print(f"\nTotal: {len(occlusion_df)} slices, {occlusion_df['has_meaningful_padding'].sum()} with meaningful padding to occlude")
print(f"Mean baseline logit: {occlusion_df['baseline_logit'].mean():.2f} (mean baseline P: {occlusion_df['baseline_p'].mean():.4f} -- "
      f"near-saturated, confirming logit is the metric to trust here)")

[fold 0] occlusion sensitivity: 10 slices done


[fold 1] occlusion sensitivity: 16 slices done


[fold 2] occlusion sensitivity: 14 slices done


[fold 3] occlusion sensitivity: 13 slices done


[fold 4] occlusion sensitivity: 12 slices done

Total: 65 slices, 58 with meaningful padding to occlude
Mean baseline logit: 12.95 (mean baseline P: 0.9992 -- near-saturated, confirming logit is the metric to trust here)


In [18]:
from scipy.stats import wilcoxon

occlusion_out_csv = RESULTS_IMAGING_DIR / "confound_check_occlusion_sensitivity.csv"
occlusion_df.to_csv(occlusion_out_csv, index=False)
print(f"Wrote {occlusion_out_csv} ({len(occlusion_df)} rows)")

mean_abs_delta_tumor = occlusion_df["delta_tumor_logit"].abs().mean()
mean_abs_delta_padding = occlusion_df["delta_padding_logit"].abs().mean()  # NaN-skipping by default
mean_abs_delta_control = occlusion_df["delta_control_logit"].abs().mean()

print("=== Occlusion Sensitivity Summary (logit space -- see markdown above) ===")
print(f"n slices: {len(occlusion_df)}, {int(occlusion_df['has_meaningful_padding'].sum())} with meaningful padding to occlude")
print(f"Mean baseline logit: {occlusion_df['baseline_logit'].mean():.2f} (mean baseline P: {occlusion_df['baseline_p'].mean():.4f})")
print(f"\nTumor occlusion:   mean delta_logit = {occlusion_df['delta_tumor_logit'].mean():+.3f}  (mean |delta| = {mean_abs_delta_tumor:.3f})")
print(f"Padding occlusion: mean delta_logit = {occlusion_df['delta_padding_logit'].mean():+.3f}  (mean |delta| = {mean_abs_delta_padding:.3f})")
print(f"Random control:    mean delta_logit = {occlusion_df['delta_control_logit'].mean():+.3f}  (mean |delta| = {mean_abs_delta_control:.3f})")

print(f"\nTumor sensitivity vs. control:   {mean_abs_delta_tumor / mean_abs_delta_control:.2f}x")
print(f"Padding sensitivity vs. control: {mean_abs_delta_padding / mean_abs_delta_control:.2f}x")

valid_tumor = occlusion_df.dropna(subset=["delta_tumor_logit", "delta_control_logit"])
stat_t, p_tumor = wilcoxon(valid_tumor["delta_tumor_logit"].abs(), valid_tumor["delta_control_logit"].abs())
print(f"\nWilcoxon signed-rank |delta_tumor_logit| vs |delta_control_logit| (n={len(valid_tumor)}): p={p_tumor:.4f}")

valid_padding = occlusion_df.dropna(subset=["delta_padding_logit", "delta_control_logit"])
if len(valid_padding) >= 5:
    stat_p, p_padding = wilcoxon(valid_padding["delta_padding_logit"].abs(), valid_padding["delta_control_logit"].abs())
    print(f"Wilcoxon signed-rank |delta_padding_logit| vs |delta_control_logit| (n={len(valid_padding)}): p={p_padding:.4f}")
else:
    p_padding = float("nan")
    print(f"Too few slices with meaningful padding (n={len(valid_padding)}) for a Wilcoxon test")

print("\n--- For reference, the same comparison in probability space (shown to be near-saturated above) ---")
print(f"Tumor |delta_p|: {occlusion_df['delta_tumor_p'].abs().mean():.5f}, "
      f"Padding |delta_p|: {occlusion_df['delta_padding_p'].abs().mean():.5f}, "
      f"Control |delta_p|: {occlusion_df['delta_control_p'].abs().mean():.5f}")

Wrote /workspace/results/imaging/confound_check_occlusion_sensitivity.csv (65 rows)
=== Occlusion Sensitivity Summary (logit space -- see markdown above) ===
n slices: 65, 58 with meaningful padding to occlude
Mean baseline logit: 12.95 (mean baseline P: 0.9992)

Tumor occlusion:   mean delta_logit = +0.066  (mean |delta| = 0.212)
Padding occlusion: mean delta_logit = -0.041  (mean |delta| = 0.370)
Random control:    mean delta_logit = +0.151  (mean |delta| = 0.342)

Tumor sensitivity vs. control:   0.62x
Padding sensitivity vs. control: 1.08x

Wilcoxon signed-rank |delta_tumor_logit| vs |delta_control_logit| (n=65): p=0.0065
Wilcoxon signed-rank |delta_padding_logit| vs |delta_control_logit| (n=58): p=0.9168

--- For reference, the same comparison in probability space (shown to be near-saturated above) ---
Tumor |delta_p|: 0.00006, Padding |delta_p|: 0.00021, Control |delta_p|: 0.00046


In [19]:
fig, ax = plt.subplots(figsize=(6, 4))
conditions = ["tumor", "padding", "control"]
means = [mean_abs_delta_tumor, mean_abs_delta_padding, mean_abs_delta_control]
sems = [occlusion_df["delta_tumor_logit"].abs().sem(), occlusion_df["delta_padding_logit"].abs().sem(), occlusion_df["delta_control_logit"].abs().sem()]
ax.bar(conditions, means, yerr=sems, capsize=5, color=["#d62728", "#ff7f0e", "#7f7f7f"])
ax.set_ylabel("mean |Δ logit| after occlusion")
ax.set_title(f"Occlusion sensitivity, logit space (n={len(occlusion_df)} slices)\np(tumor vs control)={p_tumor:.3f}, p(padding vs control)={p_padding:.3f}")
plt.tight_layout()
occlusion_fig_path = EVAL_IMAGING_DIR / "confound_check_occlusion_sensitivity.png"
fig.savefig(occlusion_fig_path, dpi=110)
plt.close(fig)
print(f"Saved {occlusion_fig_path}")

print("\n=== Occlusion Verdict (logit space) ===")
ALPHA = 0.05
tumor_significant = p_tumor < ALPHA
padding_significant = (not np.isnan(p_padding)) and p_padding < ALPHA
tumor_bigger = mean_abs_delta_tumor > mean_abs_delta_control
padding_bigger = mean_abs_delta_padding > mean_abs_delta_control

if tumor_significant and tumor_bigger:
    print(f"TUMOR REGION IS CAUSALLY IMPORTANT: occluding it moves the logit by {mean_abs_delta_tumor:.3f} on average "
          f"({mean_abs_delta_tumor / mean_abs_delta_control:.2f}x the random-control baseline), "
          f"significantly more than occluding an arbitrary same-sized patch (p={p_tumor:.4f}). "
          "This is direct, method-independent evidence the model's prediction genuinely depends on the real "
          "pathology region, not just correlates with it via a shortcut elsewhere.")
elif tumor_significant and not tumor_bigger:
    print(f"Occluding the tumor region moved the logit SIGNIFICANTLY LESS than the random control "
          f"(p={p_tumor:.4f}) -- the model is, if anything, less sensitive to the real pathology region than "
          "to an arbitrary patch of tissue. This is a concerning result consistent with the model not relying "
          "on the tumor region for its decision.")
else:
    print(f"No significant difference between tumor occlusion and random control (p={p_tumor:.4f}, "
          f"{mean_abs_delta_tumor / mean_abs_delta_control:.2f}x) -- occluding the real pathology region does not "
          "move the logit meaningfully more than occluding an arbitrary same-sized patch of tissue. The model's "
          "decision does not appear to causally depend on the tumor region specifically, at least not more than it "
          "depends on comparably-sized patches of surrounding tissue in general.")

if occlusion_df["has_meaningful_padding"].sum() >= 5:
    if padding_significant and padding_bigger:
        print(f"\nPADDING IS CAUSALLY IMPORTANT: disrupting it moves the logit by {mean_abs_delta_padding:.3f} on average "
              f"({mean_abs_delta_padding / mean_abs_delta_control:.2f}x control), significantly more than a random patch "
              f"(p={p_padding:.4f}). This is direct evidence the model DOES use the padding pattern as a cue -- "
              "concerning, and consistent with (though not proof of) a scanner/dataset shortcut.")
    else:
        print(f"\nPadding occlusion does not move the logit significantly more than the random control "
              f"(p={p_padding:.4f}, {mean_abs_delta_padding / mean_abs_delta_control:.2f}x). Disrupting the padding pattern "
              "does not appear to causally matter to the prediction more than disrupting any other same-sized region -- "
              "reassuring on this specific axis, though it does not rule out the model using some other "
              "non-anatomical cue not tested here.")
else:
    print("\nToo few slices had meaningful padding to occlude for a reliable padding verdict.")

Saved /workspace/outputs/eval/imaging/confound_check_occlusion_sensitivity.png

=== Occlusion Verdict (logit space) ===
Occluding the tumor region moved the logit SIGNIFICANTLY LESS than the random control (p=0.0065) -- the model is, if anything, less sensitive to the real pathology region than to an arbitrary patch of tissue. This is a concerning result consistent with the model not relying on the tumor region for its decision.

Padding occlusion does not move the logit significantly more than the random control (p=0.9168, 1.08x). Disrupting the padding pattern does not appear to causally matter to the prediction more than disrupting any other same-sized region -- reassuring on this specific axis, though it does not rule out the model using some other non-anatomical cue not tested here.
